In [3]:
import sys
print("Python executable:", sys.executable)
print("Python version:", sys.version)

Python executable: c:\Users\Daniel\ewallet-sentiment-analysis\venv\Scripts\python.exe
Python version: 3.14.2 (tags/v3.14.2:df79316, Dec  5 2025, 17:18:21) [MSC v.1944 64 bit (AMD64)]


In [4]:
%pip install pandas matplotlib seaborn tqdm langdetect emoji google-play-scraper

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import pandas as pd
import matplotlib.pyplot as plt
print("✅ Libraries loaded")
print(f"Pandas version: {pd.__version__}")

df = pd.read_csv("reviews_raw_combined_20260605.csv")
print(f"✅ Loaded {len(df):,} reviews")
df.head(3)

✅ Libraries loaded
Pandas version: 3.0.3
✅ Loaded 248,858 reviews


,app_name,scraped_lang,userName,score,content,thumbsUpCount,reviewCreatedVersion,at,replyContent,repliedAt
0,Touch n Go eWallet,en,A Google user,5,apps yg sngt berguna,0,1.9.0,2026-06-04 16:02:02,NaN,NaN
1,Touch n Go eWallet,en,A Google user,5,good,0,1.9.0,2026-06-04 15:53:41,NaN,NaN
2,Touch n Go eWallet,en,A Google user,5,nice app,0,1.8.98,2026-06-04 12:59:24,NaN,NaN


In [6]:
#Full audit of dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option('display.max_colwidth', 200)
sns.set_style("whitegrid")

print(f"Shape: {df.shape}")
print(f"\nColumns: {df.columns.tolist()}")
print(f"\nMissing values per column:")
print(df.isnull().sum())
print(f"\nData types:")
print(df.dtypes)

Shape: (248858, 10)

Columns: ['app_name', 'scraped_lang', 'userName', 'score', 'content', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt']

Missing values per column:
app_name                    0
scraped_lang                0
userName                    0
score                       0
content                    14
thumbsUpCount               0
reviewCreatedVersion    45155
at                          0
replyContent            79634
repliedAt               79634
dtype: int64

Data types:
app_name                  str
scraped_lang              str
userName                  str
score                   int64
content                   str
thumbsUpCount           int64
reviewCreatedVersion      str
at                        str
replyContent              str
repliedAt                 str
dtype: object


In [7]:
#read real samples
for app in df["app_name"].unique():
    print(f"\n{'='*60}")
    print(f"📱 {app}")
    print('='*60)
    sample = df[df["app_name"] == app].sample(3, random_state=42)
    for _, row in sample.iterrows():
        print(f"\n[⭐ {row['score']}] [{row['scraped_lang']}] {row['content'][:200]}")


📱 Touch n Go eWallet

[⭐ 1] [ms] Payah jugak aku dah tukar no pastu bila download balik app ni nak login takleh sebab acc aku connect dengan no lama aku. Sekarang nak create acc baru pun takleh

[⭐ 5] [en] mudah

[⭐ 1] [en] why after update i cannot topup my card using nfc anymore. ayyy please fix this

📱 MAE by Maybank

[⭐ 5] [ms] Ok

[⭐ 2] [en] Kenapa Mae ni asyik kena masuk password dua tiga kali padahal dah log in awal2

[⭐ 1] [en] Have to try three times to get the app to give me the secure2u verification, even after going into the secure2u section of the app. What a waste of time.

📱 Setel

[⭐ 4] [en] 👏🏻👏🏻💪💪🤲🤲🎉🎉

[⭐ 5] [en] memudahkn pengguna

[⭐ 5] [en] Terbaik

📱 Boost

[⭐ 3] [ms] Ok

[⭐ 3] [en] i suppose to get a 50% rebate but i still didn't get it...please fix it soon

[⭐ 5] [en] good job

📱 GXBank

[⭐ 1] [en] U can't even reinstall the apps and login.. so many issues, i don't even recounts how many times i need to take selfies just to try login.. the worst part is the sugg

In [8]:
# --- Data Cleaning ---
print(f"Before cleaning: {len(df):,}")

# 1. Drop rows with no review text
df = df.dropna(subset=["content"])
print(f"After dropping null content: {len(df):,}")

# 2. Strip whitespace
df["content"] = df["content"].astype(str).str.strip()

# 3. Remove duplicates (same user + same content)
df = df.drop_duplicates(subset=["userName", "content"], keep="first")
print(f"After removing duplicates: {len(df):,}")

# 4. Remove very short reviews (< 3 words)
df["word_count"] = df["content"].str.split().str.len()
df = df[df["word_count"] >= 3]
print(f"After removing <3 word reviews: {len(df):,}")

# 5. Parse review date
df["at"] = pd.to_datetime(df["at"], errors="coerce")
df["year"] = df["at"].dt.year
df["year_month"] = df["at"].dt.to_period("M").astype(str)

# 6. Reset index
df = df.reset_index(drop=True)

print(f"\nFinal cleaned shape: {df.shape}")
print(f"Date range: {df['at'].min()} → {df['at'].max()}")

Before cleaning: 248,858
After dropping null content: 248,844
After removing duplicates: 161,135
After removing <3 word reviews: 137,182

Final cleaned shape: (137182, 13)
Date range: 2016-12-15 14:21:26 → 2026-06-04 16:27:39


In [9]:
# Language detection on sample (test first)
from langdetect import detect, DetectorFactory
DetectorFactory.seed = 42

def detect_lang_safe(text):
    try:
        return detect(text)
    except:
        return "unknown"

sample = df.sample(5000, random_state=42).copy()
sample["detected_lang"] = sample["content"].apply(detect_lang_safe)

print(sample["detected_lang"].value_counts().head(10))

detected_lang
en    2790
id    1751
tl      91
af      50
so      32
no      29
fr      28
da      21
sl      21
cy      20
Name: count, dtype: int64


In [ ]:
#Full language detection on entire dataset (with progress bar)
from tqdm import tqdm
tqdm.pandas()

df["detected_lang"] = df["content"].progress_apply(detect_lang_safe)

def simplify_lang(lang):
    if lang == "en":
        return "english"
    elif lang in ["ms", "id"]:
        return "malay"
    else:
        return "other"

df["lang_group"] = df["detected_lang"].apply(simplify_lang)
print(df["lang_group"].value_counts())

100%|██████████| 137182/137182 [2:56:51<00:00, 12.93it/s]    

lang_group
english    77744
malay      46789
other      12649
Name: count, dtype: int64


In [11]:
#Save cleaned dataset
df.to_csv("reviews_cleaned.csv", index=False)
print(f"✅ Saved {len(df):,} cleaned reviews")

✅ Saved 137,182 cleaned reviews
